# M06 | Final dataset verification and analytical handover
**NHS A&E Intelligent Decision Support System**

## Research objective
Can the prepared provider-month calendar be reproduced from the verified historical data, and can we preserve the published counts while creating a consistent dataset for time-aware forecasting?

This notebook verifies and publishes the completed M05 calendar. It does not construct model features, impute values, split training data or fit models. Source: [NHS A&E IDSS repository](https://github.com/neoscar10/NHS-A-E-IDSS).

## 1. Setup and source selection
The release is tied to a completed M05 run and its exact M04 inputs. By default, the most recent M05 run is selected. Set `M05_RUN_ID` to an earlier verified run to reproduce it.

In [ ]:
from pathlib import Path
import sys
import pandas as pd
from IPython.display import display, Markdown, Image

repo_root = next((p for p in (Path.cwd(), *Path.cwd().parents)
                  if (p / "src/data/m06_finalise.py").is_file()), None)
if repo_root is None:
    raise FileNotFoundError("Open the NHS-A-E-IDSS project in VS Code.")
sys.path.insert(0, str(repo_root))
from src.data.m06_finalise import run_finalise
M05_RUN_ID = None  # Set a verified run ID to reproduce an earlier release.


## 2. Execute final dataset verification
The verification reconstructs the calendar from M04 and compares it with M05, checks the original provider observations, reconciles 18 measures across all reporting months and checks the exported file formats. The evidence exporter saves the complete results and regression test log.

In [ ]:
result = run_finalise(repo_root, m05_run_id=M05_RUN_ID, notebook=True)
s = result["summary"]
if result["evidence_status"] != "COMPLETE":
    raise RuntimeError("Review the evidence manifest before using this release.")
print("Verification:", s["status"])
print("Evidence status:", result["evidence_status"])
print("Release ID:", s["run_id"])
print("M05 source run:", s["m05_run_id"])
print("Format:", s["parquet_status"])
display(pd.DataFrame({"Metric": ["Reporting months", "Distinct provider codes", "Observed provider rows", "Absent calendar rows", "Calendar rows", "National totals", "Reconciliation comparisons", "Mismatches"],
                      "Value": [s[k] for k in ["reporting_months", "provider_codes", "observed_rows", "absent_rows", "calendar_rows", "national_rows", "national_comparisons", "national_mismatches"]]}))

## 3. Record conservation and calendar structure
**Research question:** Does the final calendar retain all published observations while representing absent months explicitly?

The chart separates reported provider-month records from calendar rows with no reported observation. These additional rows are not additional attendances or submissions.

In [ ]:
evidence = Path(result["evidence_path"])
def show_figure(name):
    display(Image(filename=str(evidence / "figures" / name), width=950))
show_figure("01_final_record_coverage.png")
display(Markdown(f"**Finding:** {s['observed_rows']:,} observed records and {s['absent_rows']:,} absent rows form the {s['calendar_rows']:,}-row calendar. **Decision:** retain the month-aligned calendar for feature engineering; do not convert unknown counts into zero."))

## 4. Forecasting candidate coverage
**Research question:** How much of the reported provider population satisfies the provisional origin-specific history rule?

The rule requires an observed origin, a complete history window and positive Type 1 activity within that window. It is an activity proxy rather than verified service eligibility.

In [ ]:
show_figure("02_candidate_coverage.png")
display(Markdown(f"**Finding:** In {s['latest_month']}, {s['latest_candidates']:,} of {s['latest_observed_providers']:,} reported providers satisfy the {s['history_window_months']}-month rule. Across the series there are {s['candidate_origin_rows']:,} candidate origins. **Decision:** use origin-specific eligibility, without filtering by future outcomes. Candidate origins are not yet model training examples."))

## 5. National reconciliation
**Research question:** Do the observed provider counts reproduce the published aggregate series?

The two lines should overlap when reconciliation succeeds. National totals are retained as benchmarks and excluded from the provider calendar.

In [ ]:
show_figure("03_national_type1_reconciliation.png")
display(Markdown(f"**Finding:** {s['national_comparisons']:,} comparisons across all {s['original_measures']} measures produced {s['national_mismatches']:,} differences. **Decision:** keep national totals separate from provider-level forecasting records. This establishes arithmetic agreement, not independent confirmation of every NHS submission."))

## 6. Dataset contract and sample
The calendar has a composite key of `org_code` and `period`. `type1_attendances` is the outcome measure, not a future target column yet. Absent rows have null counts and no fabricated source metadata.

In [ ]:
cols = ["period", "org_code", "type1_attendances", "is_observed", "observation_state", "type1_candidate"]
display(result["panel"][cols].head(8))
absent = result["panel"].loc[~result["panel"].is_observed, cols]
if not absent.empty:
    display(absent.head(5))
display(result["dictionary"].tail(8))

## 7. Evidence-based decisions for M07
The next step is feature engineering. Lag features must respect actual calendar intervals, publication availability and chronological evaluation. An absent future target cannot be scored as zero.

In [ ]:
display(result["decisions"])
print("Dataset release:", Path(result["output_path"]).relative_to(repo_root))
print("Research evidence:", evidence.relative_to(repo_root))

## Conclusion
The final release establishes data integrity and reproducibility. It is not a claim of operational forecasting validity. Historical revisions, source publication timing and provider identity changes require consideration in the modelling design. No predictive performance has been measured.